In [128]:
import numpy as np

text = "good morning"

chars = sorted(list(set(text)))

char_to_idx = {ch: i for i, ch in enumerate(chars)}

idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(char_to_idx)
print(idx_to_char)

text_indices = [char_to_idx[ch] for ch in text]

print(text_indices)

{' ': 0, 'd': 1, 'g': 2, 'i': 3, 'm': 4, 'n': 5, 'o': 6, 'r': 7}
{0: ' ', 1: 'd', 2: 'g', 3: 'i', 4: 'm', 5: 'n', 6: 'o', 7: 'r'}
[2, 6, 6, 1, 0, 4, 6, 7, 5, 3, 5, 2]


In [129]:
sequence_length = 1

X = []
y = []

for i in range(len(text_indices) - sequence_length):
    X.append(text_indices[i:i+sequence_length])

    y.append(text_indices[i+sequence_length])


X = np.array(X)
y = np.array(y)
print(X)
print(y)

[[2]
 [6]
 [6]
 [1]
 [0]
 [4]
 [6]
 [7]
 [5]
 [3]
 [5]]
[6 6 1 0 4 6 7 5 3 5 2]


In [130]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import SimpleRNN, Dense

model = Sequential()

model.add(SimpleRNN(128, input_shape=(X.shape[1], len(chars)),
activation='relu'))

model.add(Dense(len(chars), activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
metrics=['accuracy'])

In [131]:
X_onehot = np.zeros((X.shape[0], X.shape[1], len(chars)))

for i, sequence in enumerate(X):
    for t, char_idx in enumerate(sequence):
        X_onehot[i, t, char_idx]

print(X_onehot)

y = np.expand_dims(y, axis=-1)

[[[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0.]]]


In [132]:
model.fit(X_onehot, y, epochs=5000, batch_size=32)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.0909 - loss: 2.0794
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0790
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0786
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2727 - loss: 2.0782
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2727 - loss: 2.0778
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0774
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0770
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2727 - loss: 2.0766
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0762
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0758
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2727 - loss: 2.0754
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2727 - loss: 2.0750


In [133]:
def predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length):
    if len(seed) < sequence_length:
        seed = seed.zfill(sequence_length)
    elif len(seed) > sequence_length:
        seed = seed[-sequence_length:]

    seed_indices = [char_to_idx[ch] for ch in seed]

    X_seed = np.zeros((1, sequence_length, len(char_to_idx)))
    for t, idx in enumerate(seed_indices):
        X_seed[0, t, idx]

    prediction = model.predict(X_seed, verbose=0)

    predicted_idx = np.argmax(prediction, axis=-1)[0]

    predicted_char = idx_to_char[predicted_idx]

    return predicted_char

In [134]:
seed = "good m"

predicted_char = predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length)

print(f"Predicted next character: {predicted_char}")

Predicted next character: o
